# Case 1: 50keV Cascades in Fe

This is a validation benchmark for comparison to data from 
> K. Nordlund et al., “Primary radiation damage in materials: Review of current understanding and proposed new standard displacement damage model to incorporate in cascade defect production efficiency and mixing effects,” OECD Nuclear Energy Agency, Paris, Jan. 2015.

In page 61, data are given for damage in Fe 78.7 keV PKA cascade which corresponds to 50 keV damage energy. The data are taken from MD simulations by Stoller, SRIM simulations and arc-dpa model. 

| Simulation Method         | Number of defects |
| ------------------------- | ----------------- |
| MD 50keV, 100K            | 168 ± 4           |
| NRT model 50keV           | 500               |
| SRIM QC 78.7keV           | 530               |
| arc-dpa + SRIM QC 78.7keV | 218               |

Cascade simulations are performed with OpenTRIM for 2 initial energy settings:
- $E_0=78.7$ keV, electronic stopping on
- $E_0=50.0$ keV, electronic stopping off

Both off these cases are run with and without defect recombination.  

### 1. Import libs, set Model Parameters

In [1]:
import numpy as np
import opentrim
print('opentrim', opentrim.__version__)

a0 = 0.28665    # Fe lattice const [nm]
rho_fe = 7.8658 # Fe mass density
L = 1000.0     # simulation box size [nm]
Ed = 40.0       # Fe Displacement enrgy [eV]
Ec = 10.0       # Ion energy lower cut-off [eV]
Rc = 2.8*a0     # Fe defect recombination radius (taken from Ortiz2020) 
E0 = [50000.0, 78700.0]    # Initial PKA enrgy
Stop = [opentrim.Stopping.Off, opentrim.Stopping.SRIM13] # stopping code
Recomb = [False, True] # enable recombination?
Nh = 1000      # Histories to run

opentrim 1.2.0


### 2. Setup OpenTRIM Config

In [2]:
config = opentrim.Config()

config.Simulation.simulation_type = opentrim.SimulationType.CascadesOnly
config.Simulation.electronic_stopping = opentrim.Stopping.Off

config.Transport.flight_path_type = opentrim.FlightPath.Variable
config.Transport.min_energy = Ec

config.IonBeam.ion = opentrim.Element('Fe')
config.IonBeam.spatial_distribution.geometry = opentrim.Geometry.Volume
config.IonBeam.spatial_distribution.center = np.array([0.5, 0.5, 0.5])*L
config.IonBeam.angular_distribution.type = opentrim.Distribution.Uniform
config.IonBeam.angular_distribution.fwhm = 4*np.pi # Isotropic 4π

config.Target.size = np.array([1,1,1])*L
config.Target.periodic_bc = [0,0,0]

# material
mat = opentrim.Material(); mat.id = 'Fe'; mat.density = rho_fe
atom = opentrim.Atom(); atom.element = opentrim.Element('Fe')
atom.X = 1.0; atom.Ed = Ed; atom.El = 0.001; atom.Er = Ed; atom.Rc = Rc
mat.composition.append(atom)
config.Target.materials.append(mat)
#region
region = opentrim.Region(); region.id = 'cube'; region.material_id = 'Fe'
region.size = np.array([1,1,1])*L
config.Target.regions.append(region)

# Run
config.Run.max_no_ions = Nh

config.validate() # check it
config # show it

Config(ion=Fe, max_no_ions=1000, threads=0)

### 3. Run all cases

In [ ]:
# arrays to recieve results
Nv = np.zeros((len(E0),len(Recomb)))
NRT = np.zeros((len(E0),len(Recomb)))

# run all cases in series
for i in range(len(E0)):
    for j in range(len(Recomb)):
        print(f"{i*2 + j + 1:d}/4.","E=",E0[i],"Recomb=",Recomb[j])
        # prepare config
        config.Simulation.electronic_stopping = Stop[i]
        config.Simulation.defect_recombination = Recomb[j]
        config.IonBeam.energy_distribution.center = E0[i]
        # get a simulation driver and run it
        D = opentrim.Driver(config)
        D.run()
        D.wait() # wait for results
        # get the total tally
        info = opentrim.Info(D)
        x, dx = info['tally']['totals']['data']
        Nv[i,j] = x[1,1]
        NRT[i,j] = x[14,1]

1/4. E= 50000.0 Recomb= False
2/4. E= 50000.0 Recomb= True
3/4. E= 78700.0 Recomb= False
4/4. E= 78700.0 Recomb= True


### 4. Print results


In [6]:

print('50.0keV, no stopping        ', f"{Nv[0,0]:.0f}")
print('50.0keV, no stopping, Recomb', f"{Nv[0,1]:.0f}")
print('78.7keV, NRT-LSS approx     ', f"{NRT[1,0]:.0f}")
print('78.7keV                     ', f"{Nv[1,0]:.0f}")
print('78.7keV, Recomb             ', f"{Nv[1,1]:.0f}")

50.0keV, no stopping         528
50.0keV, no stopping, Recomb 166
78.7keV, NRT-LSS approx      507
78.7keV                      646
78.7keV, Recomb              197
